In [11]:
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast

# secret shhhhh
def get_secret():

    secret_name = "gitlab/tcwyu-respository"
    region_name = "us-east-2"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return secret
    
secret = ast.literal_eval(get_secret())


In [19]:


# environment = {
#     SM_NUM_GPUS = 2, # ?
#     #SM_MODEL_DIR = "output",
#     #SM_OUTPUT_DATA_DIR = "output", # relative to git repo, or new folder in S3?
#     SM_CHANNEL_TRAIN = "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/train",
#     SM_CHANNEL_TEST = "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/test",
#     SM_TOKENIZER = "s3://himss2023mimiciv/uw_skynet/data/tokenizer-mimic-iv-icd-final",
#     SM_VSA_PARSER = "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/icd_to_vsa_data.pkl",
#     SM_MODEL_CHKPT = "s3://himss2023mimiciv/uw_skynet/MODEL_CHECKPOINT"
# }

# git config
git_config = {
    'repo': 'https://git.uwaterloo.ca/medical-ai-lab/bertha_experiments.git',
    'username': secret['username'],
    'password': secret['password']
}

hyperparameters = {
    'epochs': 5, 
    'batch_size': 80, 
    'lr': 2.5e-5
}



estimator_args = {
    "role":"arn:aws:iam::776029861160:role/service-role/AmazonSageMaker-ExecutionRole-20230328T101238",
    "output_path":"s3://himss2023mimiciv/uw_skynet/output",
    "image_uri":"776029861160.dkr.ecr.us-east-2.amazonaws.com/smile-uw-containers:medair-huggingface-pytorch-training-test",
    "model_uri":"s3://himss2023mimiciv/uw_skynet/MODEL_CHECKPOINT",
    "instance_count":1,
    "instance_type":"ml.t3.medium",
    "entry_point":"BERTHA_AWS_Mortality.py",
    "git_config":git_config
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args)

In [20]:
pytorch_estimator.fit()

Cloning into '/tmp/tmp2s_tlipk'...


ValueError: Entry point does not exist in the repo.

In [3]:
import os
os.environ

environ{'NV_CUDA_COMPAT_PACKAGE': 'cuda-compat-11-7',
        'PYTHONIOENCODING': 'UTF-8',
        'HOROVOD_VERSION': '0.26.1',
        'EFA_VERSION': '1.19.0',
        'HOSTNAME': 'f8fce8eca203',
        'LD_LIBRARY_PATH': '/opt/conda/lib/python3.9/site-packages/smdistributed/dataparallel/lib:/opt/amazon/openmpi/lib/:/opt/amazon/efa/lib/:/opt/conda/lib:/usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/lib',
        'HOME': '/root',
        'TORCH_NVCC_FLAGS': '-Xfatbin -compress-all',
        'PYTHONUNBUFFERED': '1',
        'BRANCH_OFI': '1.4.0-aws',
        'CUDA_VERSION': '11.7.1',
        'TORCH_CUDA_ARCH_LIST': '3.7 5.0 7.0+PTX 8.0',
        'PYTHONDONTWRITEBYTECODE': '1',
        'NVIDIA_REQUIRE_CUDA': 'cuda>=11.7 brand=tesla,driver>=450,driver<451 brand=tesla,driver>=470,driver<471 brand=unknown,driver>=470,driver<471 brand=nvidia,driver>=470,driver<471 brand=nvidiartx,driver>=470,driver<471 brand=geforce,driver>=470,driver<471 brand=geforcertx,driver>=470